# 06 — Monitoring, data drift et boucle de feedback

Objectif : montrer comment la performance et la distribution des données sont suivies après déploiement.


In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

print("Project root :", ROOT)
print("Raw data     :", DATA_RAW)


Project root : C:\Users\Admin\Desktop\predictmaint-ai
Raw data     : C:\Users\Admin\Desktop\predictmaint-ai\data\raw


### Analyse du résultat

Les chemins identifient explicitement la référence, les modèles et les données surveillées. Le monitoring peut ainsi être rejoué avec la même configuration que l'entraînement.


## 1. Chargement de la référence de production


In [3]:
REFERENCE_DIR = ROOT / "data" / "reference"
reference_all = pd.read_csv(REFERENCE_DIR / "reference_features.csv")
selected = json.loads((MODELS_DIR / "selected_features.json").read_text(encoding="utf-8"))
selected = [c for c in selected if c in reference_all.columns]
reference = reference_all[selected]

print("Référence :", reference.shape)


Référence : (5000, 40)


### Analyse du résultat

La référence de production contient 5 000 observations sur les 40 variables retenues. Elle constitue le profil de comparaison ; sa période et sa version doivent rester figées et documentées lors d'un vrai déploiement.


## 2. Démonstration simple du PSI sur une variable


In [4]:
def psi_1d(expected, actual, bins=10, eps=1e-6):
    expected = pd.Series(expected).dropna().astype(float)
    actual = pd.Series(actual).dropna().astype(float)

    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf

    e = pd.cut(expected, edges).value_counts(normalize=True, sort=False).clip(lower=eps)
    a = pd.cut(actual, edges).value_counts(normalize=True, sort=False).clip(lower=eps)
    return float(((a - e) * np.log(a / e)).sum())

feature = selected[0]
current = reference[feature].sample(min(1200, len(reference)), random_state=7).copy()
shift = float(reference[feature].std()) * 3
current_shifted = current + shift

print("Feature :", feature)
print("PSI sans drift simulé :", psi_1d(reference[feature], current))
print("PSI avec drift simulé :", psi_1d(reference[feature], current_shifted))


Feature : sensor_11_mean_5
PSI sans drift simulé : 0.006319619095611536
PSI avec drift simulé : 11.222021608252723


### Analyse du résultat

L'échantillon non modifié donne un PSI de 0,006, compatible avec de simples fluctuations d'échantillonnage. Le décalage simulé produit 11,22, très au-dessus d'un seuil d'alerte usuel : le test réagit donc fortement à une dérive volontaire.


## 3. Rapport de drift industrialisé


In [5]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.monitoring.drift import statistical_drift_report

current_df = reference.sample(min(1200, len(reference)), random_state=7).copy()
for c in selected[:max(1, len(selected) // 4)]:
    std = float(reference[c].std())
    current_df[c] = current_df[c] + (4 * std if std > 0 else 1)

report = statistical_drift_report(current_df, reference=reference)
print("Status       :", report["status"])
print("Drifted share:", report["drifted_share"])
print("Top drifts   :")
print(sorted(report["drifted_features"].items(), key=lambda x: x[1], reverse=True)[:10])


Status       : alert
Drifted share: 0.25
Top drifts   :
[('sensor_11_min_5', 12.43469178809636), ('sensor_14', 12.433856685847195), ('sensor_3_mean_10', 12.43385628584293), ('sensor_4_mean_5', 12.433855885838662), ('sensor_4_max_5', 12.429112786001713), ('sensor_4_min_5', 12.424357294557876), ('sensor_11_mean_5', 12.41489501491767), ('sensor_11_max_5', 12.370804156029944), ('sensor_17_mean_10', 12.336728238017518), ('sensor_7_max_20', 11.645046851176618)]


### Analyse du résultat

Le rapport passe en statut `alert` avec 25 % des variables signalées. Les PSI proches de 12 concernent précisément les variables artificiellement décalées ; ce scénario valide le déclenchement technique, mais ne mesure pas une dérive réelle de production.


## 4. Performance différée

En production, une prédiction est faite à `t0`, mais la vérité terrain n'est connue que plus tard. `/feedback` rattache donc le résultat réel à la prédiction initiale. Le monitoring recalcule les métriques **par version de modèle et avec le seuil réellement utilisé lors de la prédiction**.


In [6]:
from src.monitoring.performance import performance_report

print("Fonction de production disponible : performance_report(...)")
print("Exemple : performance_report('/tmp/predictions.jsonl', '/tmp/feedback.jsonl')")


Fonction de production disponible : performance_report(...)
Exemple : performance_report('/tmp/predictions.jsonl', '/tmp/feedback.jsonl')


### Analyse du résultat

La fonction de performance différée est disponible, mais aucune métrique réelle n'est calculée ici faute de journaux de prédictions et de feedback. En production, la performance ne pourra être jugée qu'après rattachement des vérités terrain aux prédictions historiques.


## 5. Boucle MLOps

```text
Nouvelles données
      ↓
Qualité / schéma
      ↓
Drift + performance différée
      ↓
Guardrails dépassés ?
      ↓
Retraining sur TRAIN enrichi
      ↓
Quality gate
      ↓
Champion / Challenger
      ↓
Déploiement versionné
```

Le holdout NASA reste hors de cette boucle.
